# Explorando BPE — Byte Pair Encoding

Vamos a explorar el algoritmo BPE paso a paso ANTES de implementarlo en código de producción.

**Plan:**
1. Texto → bytes UTF-8
2. Encontrar el par más frecuente
3. Mergear ese par
4. Repetir hasta tener el vocabulario deseado
5. Visualizar qué aprendió

In [ ]:
text = "el gato come pescado. el gato come atún. el gato come."
text_bytes = text.encode("utf-8")

print(f"Texto original ({len(text)} caracteres):")
print(repr(text))
print()
print(f"Bytes UTF-8 ({len(text_bytes)} bytes):")
print(list(text_bytes))

Texto original (54 caracteres):
'el gato come pescado. el gato come atún. el gato come.'

Bytes UTF-8 (55 bytes):
[101, 108, 32, 103, 97, 116, 111, 32, 99, 111, 109, 101, 32, 112, 101, 115, 99, 97, 100, 111, 46, 32, 101, 108, 32, 103, 97, 116, 111, 32, 99, 111, 109, 101, 32, 97, 116, 195, 186, 110, 46, 32, 101, 108, 32, 103, 97, 116, 111, 32, 99, 111, 109, 101, 46]


In [2]:
# Los IDs empiezan siendo los bytes (cada byte = un token inicialmente)
ids = list(text_bytes)
print(f"Cantidad inicial de tokens: {len(ids)}")

# Encontrar el par de IDs adyacentes más frecuente
from collections import Counter

# Truco clave: zip(ids, ids[1:]) genera los pares adyacentes
# Ejemplo: [101, 108, 32] → [(101, 108), (108, 32)]
pairs = zip(ids, ids[1:])
pair_counts = Counter(pairs)

# Mostrar los 5 más frecuentes
print("\nTop 5 pares más frecuentes:")
for pair, count in pair_counts.most_common(5):
    pair_chars = ''.join(chr(b) if 32 <= b < 127 else '?' for b in pair)
    print(f"  {pair}  ('{pair_chars}')  →  {count} veces")


Cantidad inicial de tokens: 55

Top 5 pares más frecuentes:
  (97, 116)  ('at')  →  4 veces
  (101, 108)  ('el')  →  3 veces
  (108, 32)  ('l ')  →  3 veces
  (32, 103)  (' g')  →  3 veces
  (103, 97)  ('ga')  →  3 veces


In [3]:
# Helper: la función merge reemplaza todas las ocurrencias de un par por un nuevo ID
def merge(ids, pair, new_id):
    new_ids = []
    i = 0
    while i < len(ids):
        # Si el par está en la posición actual, lo mergeamos
        if ids[i] == pair[0] and i + 1 < len(ids) and ids[i+1] == pair[1]:
            new_ids.append(new_id)
            i += 2  # ¡clave! saltamos los DOS elementos consumidos
        else:
            new_ids.append(ids[i])
            i += 1  # solo saltamos UNO
    return new_ids

# Encontrar el par más frecuente
most_frequent_pair = max(pair_counts, key=pair_counts.get)
print(f"Par más frecuente: {most_frequent_pair}")

# Helper: la función merge reemplaza todas las ocurrencias de un par por un nuevo ID
def merge(ids, pair, new_id):
    new_ids = []
    i = 0
    while i < len(ids):
        # Si el par está en la posición actual, lo mergeamos
        if ids[i] == pair[0] and i + 1 < len(ids) and ids[i+1] == pair[1]:
            new_ids.append(new_id)
            i += 2  # ¡clave! saltamos los DOS elementos consumidos
        else:
            new_ids.append(ids[i])
            i += 1  # solo saltamos UNO
    return new_ids

# Encontrar el par más frecuente
most_frequent_pair = max(pair_counts, key=pair_counts.get)
print(f"Par más frecuente: {most_frequent_pair}")

# Mergear — 256 es el primer ID nuevo (0-255 son bytes)
new_id = 256
ids_after = merge(ids, most_frequent_pair, new_id)

print(f"\nTokens antes:    {len(ids)}")
print(f"Tokens después:  {len(ids_after)}")
print(f"\nPrimeros 20 IDs antes:    {ids[:20]}")
print(f"Primeros 20 IDs después:  {ids_after[:20]}")


Par más frecuente: (97, 116)
Par más frecuente: (97, 116)

Tokens antes:    55
Tokens después:  51

Primeros 20 IDs antes:    [101, 108, 32, 103, 97, 116, 111, 32, 99, 111, 109, 101, 32, 112, 101, 115, 99, 97, 100, 111]
Primeros 20 IDs después:  [101, 108, 32, 103, 256, 111, 32, 99, 111, 109, 101, 32, 112, 101, 115, 99, 97, 100, 111, 46]


In [ ]:
# Loop completo: repetimos el find-pair + merge muchas veces
num_merges = 20
ids = list(text_bytes)  # reset a los IDs originales
merges = {}  # (int, int) → int: aprendemos este diccionario

for i in range(num_merges):
    # 1. Encontrar el par más frecuente
    pair_counts = Counter(zip(ids, ids[1:]))
    pair = max(pair_counts, key=pair_counts.get)

    # 2. Mergear con un nuevo ID
    new_id = 256 + i
    ids = merge(ids, pair, new_id)

    # 3. Guardar el merge (esto es lo que después usa encode)
    merges[pair] = new_id

    pair_chars = ''.join(chr(b) if 32 <= b < 127 else '?' for b in pair)
    print(f"Merge {i+1:2d}/{num_merges}: {pair} ('{pair_chars}') → {new_id}  |  tokens restantes: {len(ids)}")

print(f"\n✓ Aprendimos {len(merges)} merges")
print(f"  Vocabulario: 256 (bytes) + {len(merges)} (merges) = {256 + len(merges)}")
print(f"  Texto original: {len(text_bytes)} bytes")
print(f"  Texto tokenizado: {len(ids)} tokens")
print(f"  Compresión: {len(ids) / len(text_bytes) * 100:.1f}% del tamaño original")

In [5]:
# Texto NUEVO (distinto al que usamos para aprender los merges)
test_text = "el gato come pescado"
test_bytes = test_text.encode("utf-8")
test_ids = list(test_bytes)

print(f"Texto: '{test_text}'")
print(f"Tokens iniciales (bytes): {test_ids}")
print(f"Cantidad inicial: {len(test_ids)}")

# Función encode: aplica los merges aprendidos al texto nuevo
def encode_with_merges(ids, merges):
    while len(ids) >= 2:
        pair_counts = Counter(zip(ids, ids[1:]))
        # CLAVE: aplicamos el merge con el ID más bajo (= aprendido primero)
        pair = min(pair_counts, key=lambda p: merges.get(p, float('inf')))
        if pair not in merges:
            break
        ids = merge(ids, pair, merges[pair])
    return ids

encoded = encode_with_merges(test_ids, merges)

print(f"\nTokens finales: {encoded}")
print(f"Cantidad final: {len(encoded)}")
print(f"Compresión: {len(encoded) / len(test_ids) * 100:.1f}%")

Texto: 'el gato come pescado'
Tokens iniciales (bytes): [101, 108, 32, 103, 97, 116, 111, 32, 99, 111, 109, 101, 32, 112, 101, 115, 99, 97, 100, 111]
Cantidad inicial: 20

Tokens finales: [275]
Cantidad final: 1
Compresión: 5.0%


In [6]:
vocab = {i: bytes([i]) for i in range(256)}
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]   # ¡composición recursiva!

# Función decode: IDs → bytes → texto
def decode(ids, vocab):
    text_bytes = b"".join(vocab[i] for i in ids)
    text = text_bytes.decode("utf-8", errors="replace")
    return text

# Round-trip test: encode → decode = texto original
decoded = decode(encoded, vocab)

print(f"Texto original:     '{test_text}'")
print(f"Tokens encoded:     {encoded}")
print(f"Texto recuperado:   '{decoded}'")
print(f"\n✓ Round-trip exitoso: {test_text == decoded}")

Texto original:     'el gato come pescado'
Tokens encoded:     [275]
Texto recuperado:   'el gato come pescado'

✓ Round-trip exitoso: True
